In [ ]:
!pip install transformers==4.41.2 peft==0.11.1 trl==0.8.6 \
    bitsandbytes datasets openpyxl accelerate \
    --quiet --break-system-packages

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.9 MB/s eta 0:00:00


In [ ]:
import zipfile, os, shutil

# Clean up bad extraction
shutil.rmtree('/content/PAFT_new', ignore_errors=True)
os.makedirs('/content/PAFT_new')

# Re-extract converting Windows paths to Linux paths
with zipfile.ZipFile('/content/PAFT.zip', 'r') as z:
    for member in z.namelist():
        # Convert Windows backslashes to forward slashes
        target_path = os.path.join('/content/PAFT_new', member.replace('\\', '/'))

        if member.endswith('\\') or member.endswith('/'):
            os.makedirs(target_path, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            with z.open(member) as src, open(target_path, 'wb') as dst:
                dst.write(src.read())

print("✓ Extracted")

# Verify
import os
paft_path = '/content/PAFT_new/PAFT'
print("trainer.py exists:", os.path.exists(f'{paft_path}/src/llamafactory/train/sft/trainer.py'))
print("loader.py exists:", os.path.exists(f'{paft_path}/src/llamafactory/data/loader.py'))
print("sapaft.yaml exists:", os.path.exists(f'{paft_path}/sapaft.yaml'))

✓ Extracted
trainer.py exists: True
loader.py exists: True
sapaft.yaml exists: True


In [ ]:
paft_path = '/content/PAFT_new/PAFT'

with open(f'{paft_path}/src/llamafactory/train/sft/trainer.py') as f:
    t = f.read()
with open(f'{paft_path}/src/llamafactory/data/loader.py') as f:
    l = f.read()
with open(f'{paft_path}/src/llamafactory/hparams/finetuning_args.py') as f:
    fa = f.read()

print("SensitivitySampler:", "class SensitivitySampler" in t)
print("SAPAFTCallback:", "SAPAFTCallback" in t)
print("update_sa_weights:", "def update_sa_weights" in t)
print("on_epoch_end:", "on_epoch_end" in t)
print("use_sa_paft in finetuning_args:", "use_sa_paft" in fa)
print("compute_centroid_jsd in loader:", "compute_centroid_jsd" in l)

SensitivitySampler: True
SAPAFTCallback: True
update_sa_weights: True
on_epoch_end: True
use_sa_paft in finetuning_args: True
compute_centroid_jsd in loader: True


In [ ]:
paft_path = '/content/PAFT_new/PAFT'

# Fix Windows path to Colab path
with open(f'{paft_path}/src/llamafactory/data/loader.py') as f:
    content = f.read()

# Show current raw_path line
for i, line in enumerate(content.split('\n')):
    if 'raw_path' in line and 'hellaswag' in line:
        print(f"Current: {line.strip()}")

content = content.replace(
    'd:/PAFT/PAFT/data/hellaswag_train.json',
    '/content/PAFT_new/PAFT/data/hellaswag_train.json'
)
content = content.replace(
    'D:/PAFT/PAFT/data/hellaswag_train.json',
    '/content/PAFT_new/PAFT/data/hellaswag_train.json'
)

with open(f'{paft_path}/src/llamafactory/data/loader.py', 'w') as f:
    f.write(content)
print("✓ Path fixed")

# Also fix sapaft.yaml paths if needed
with open(f'{paft_path}/sapaft.yaml') as f:
    yaml = f.read()
print("\nsapaft.yaml:")
print(yaml)

Current: raw_path = 'd:/PAFT/PAFT/data/hellaswag_train.json'
✓ Path fixed

sapaft.yaml:
### model
model_name_or_path: Qwen/Qwen2.5-0.5B

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: q_proj,v_proj
lora_rank: 8

### dataset
dataset: hellaswag_train
dataset_dir: d:/PAFT/PAFT/data
template: qwen
cutoff_len: 512
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: d:/PAFT/PAFT/output/sapaft
logging_steps: 10
save_steps: 50000
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true

use_robust_tuning: true
use_sa_paft: true
prompt_search_space: d:/PAFT/PAFT/prompts/robust_traindataset_hel.xlsx

### eval
val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: epoch
eval_steps: 100



In [ ]:
paft_path = '/content/PAFT_new/PAFT'

for yaml_file in ['sapaft.yaml', 'paft_baseline.yaml']:
    with open(f'{paft_path}/{yaml_file}') as f:
        content = f.read()

    content = content.replace('d:/PAFT/PAFT/', f'{paft_path}/')
    content = content.replace('D:/PAFT/PAFT/', f'{paft_path}/')

    with open(f'{paft_path}/{yaml_file}', 'w') as f:
        f.write(content)
    print(f"✓ Fixed {yaml_file}")

# Verify
with open(f'{paft_path}/sapaft.yaml') as f:
    print(f.read())

✓ Fixed sapaft.yaml
✓ Fixed paft_baseline.yaml
### model
model_name_or_path: Qwen/Qwen2.5-0.5B

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: q_proj,v_proj
lora_rank: 8

### dataset
dataset: hellaswag_train
dataset_dir: /content/PAFT_new/PAFT/data
template: qwen
cutoff_len: 512
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: /content/PAFT_new/PAFT/output/sapaft
logging_steps: 10
save_steps: 50000
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true

use_robust_tuning: true
use_sa_paft: true
prompt_search_space: /content/PAFT_new/PAFT/prompts/robust_traindataset_hel.xlsx

### eval
val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: epoch
eval_steps: 100



In [ ]:
import shutil, os
paft_path = '/content/PAFT_new/PAFT'

# Clear pycache and output dirs
for root, dirs, files in os.walk(f'{paft_path}/src'):
    for d in dirs:
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

shutil.rmtree(f'{paft_path}/output/sapaft', ignore_errors=True)
os.makedirs(f'{paft_path}/output/sapaft')
shutil.rmtree(f'{paft_path}/output/paft_baseline', ignore_errors=True)
os.makedirs(f'{paft_path}/output/paft_baseline')

print("✓ Ready")

!cd {paft_path}/src && \
    WANDB_DISABLED=true \
    DISABLE_VERSION_CHECK=1 \
    PYTHONPATH={paft_path}/src \
    python train.py {paft_path}/sapaft.yaml

✓ Ready
2026-03-19 10:54:04.710683: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773917644.731728   14351 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773917644.738662   14351 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773917644.757259   14351 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773917644.757283   14351 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773917644.757289   14351 computation_placer.cc:177] computation pl

In [ ]:
paft_path = '/content/PAFT_new/PAFT'

!cd {paft_path}/src && \
    WANDB_DISABLED=true \
    DISABLE_VERSION_CHECK=1 \
    PYTHONPATH={paft_path}/src \
    python train.py {paft_path}/paft_baseline.yaml

2026-03-19 11:08:09.492937: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773918489.513494   18109 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773918489.520270   18109 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773918489.537718   18109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773918489.537742   18109 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773918489.537746   18109 computation_placer.cc:177] computation placer alr

In [ ]:
paft_path = '/content/PAFT_new/PAFT'

with open(f'{paft_path}/eval.py') as f:
    content = f.read()

# Fix the paft_path variable inside eval.py
content = content.replace(
    "paft_path = 'd:/PAFT/PAFT'",
    f"paft_path = '{paft_path}'"
)
content = content.replace(
    'paft_path = "d:/PAFT/PAFT"',
    f"paft_path = '{paft_path}'"
)

with open(f'{paft_path}/eval.py', 'w') as f:
    f.write(content)

print("✓ Fixed")

!PYTHONPATH={paft_path}/src python {paft_path}/eval.py

✓ Fixed
2026-03-19 11:21:20.489936: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773919280.511353   21648 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773919280.518722   21648 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773919280.537987   21648 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773919280.538014   21648 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773919280.538018   21648 computation_placer.cc:177] computation pl